# Query Cache — Mechanics

SLayer can cache query results **in memory, per `SlayerQueryEngine` instance**, opt-in per call
via `cache=True`. It is modelled on [Cube's in-memory cache](https://cube.dev/docs/product/caching):
a cache hit skips the database round-trip entirely, and staleness is governed by an optional
**time-to-live** and an optional set of Cube-style **refresh keys** that you scan explicitly with
`engine.refresh()`.

This notebook builds a tiny SQLite shop from scratch (so we can freely mutate it) and walks through:

1. opt-in caching (miss → hit),
2. how a hit serves the *last* result even after the data changes,
3. refresh keys invalidating on a real change,
4. TTL time-based expiry,
5. managing the cache (`evict` / `clear_cache` / `cache_size`).

> The cache is a **Python-API-only** feature — there is no REST / MCP / CLI surface. See the
> [Query Cache concept doc](../../concepts/query-cache.md) for the full reference.

In [1]:
import sqlite3
import tempfile
import time
from pathlib import Path

from slayer.async_utils import run_sync
from slayer.core.enums import DataType
from slayer.core.models import Column, DatasourceConfig, SlayerModel
from slayer.engine.cache import CacheConfig
from slayer.engine.query_engine import SlayerQueryEngine
from slayer.storage.yaml_storage import YAMLStorage

# A throwaway workspace: a SQLite "shop" database + a YAML model store.
work = Path(tempfile.mkdtemp())
db_path = work / "shop.db"

conn = sqlite3.connect(db_path)
conn.executescript(
    """
    CREATE TABLE orders (
        id         INTEGER PRIMARY KEY,
        status     TEXT,
        amount     REAL,
        updated_at TEXT
    );
    INSERT INTO orders VALUES
        (1, 'completed', 100.0, '2026-01-01'),
        (2, 'pending',    50.0, '2026-01-02'),
        (3, 'completed', 200.0, '2026-01-03');
    """
)
conn.commit()
conn.close()

def mutate(sql: str) -> None:
    """Apply a write directly to the underlying table (simulating an ETL job)."""
    c = sqlite3.connect(db_path)
    c.execute(sql)
    c.commit()
    c.close()

# Register the datasource + a simple `orders` model. Storage methods are async;
# run_sync bridges them for a notebook/script.
storage = YAMLStorage(base_dir=str(work / "store"))
run_sync(storage.save_datasource(
    DatasourceConfig(name="shop", type="sqlite", database=str(db_path))
))
run_sync(storage.save_model(SlayerModel(
    name="orders",
    sql_table="orders",
    data_source="shop",
    columns=[
        Column(name="id", sql="id", type=DataType.INT, primary_key=True),
        Column(name="status", sql="status", type=DataType.TEXT),
        Column(name="amount", sql="amount", type=DataType.DOUBLE),
        Column(name="updated_at", sql="updated_at", type=DataType.TIMESTAMP),
    ],
)))
print("workspace:", work)

workspace: /tmp/tmp8r4a7ds1


## 1. Opt-in caching

Build an engine with a `CacheConfig`. We give it two **refresh keys** on the `orders` table —
`MAX(updated_at)` and `COUNT(*)` — which we'll use in step 3. For now there is no TTL, so entries
only change when we refresh, evict, or clear them.

The query total is `100 + 50 + 200 = 350`.

In [2]:
engine = SlayerQueryEngine(
    storage=storage,
    cache_config=CacheConfig(
        ttl_seconds=None,  # no time-based expiry (yet — see step 4)
        refresh_keys=[
            ("orders", "MAX(updated_at)"),
            ("orders", "COUNT(*)"),
        ],
    ),
)

query = {"source_model": "orders", "measures": ["amount:sum"]}

first = engine.execute_sync(query, cache=True)   # miss: runs SQL, stores the result
second = engine.execute_sync(query, cache=True)  # hit: served from memory, no DB round-trip

print("first :", first.data)
print("second:", second.data)
print("cache_size:", engine.cache_size)

first : [{'orders.amount_sum': 350.0}]
second: [{'orders.amount_sum': 350.0}]
cache_size: 1


## 2. A hit serves the *last* result

Now an ETL job inserts a new completed order worth `1000`. The true total becomes `1350`.

A `cache=True` call still returns the cached `350` — that is the whole point of a cache. Passing
`cache=False` bypasses it and shows the fresh value.

In [3]:
mutate("INSERT INTO orders VALUES (4, 'completed', 1000.0, '2026-02-01')")

print("cached  (cache=True) :", engine.execute_sync(query, cache=True).data)
print("bypass  (cache=False):", engine.execute_sync(query, cache=False).data)

cached  (cache=True) : [{'orders.amount_sum': 350.0}]
bypass  (cache=False): [{'orders.amount_sum': 1350.0}]


## 3. Refresh keys — invalidate on a real change

`engine.refresh()` scans each entry's applicable refresh keys and re-executes the entry when a
value has **moved**. Our insert bumped both `MAX(updated_at)` and `COUNT(*)`, so the entry is
re-run and lands in the `refreshed` bucket. Afterwards the cached value reflects the new total.

Refresh-key sensitivity is your choice of expression: `MAX(updated_at)` catches new-timestamped
inserts; a `COUNT(*)` key additionally catches **deletes** that leave the max untouched.

In [4]:
result = engine.refresh_sync()
print("refreshed        :", len(result.refreshed))
print("expired_refreshed:", len(result.expired_refreshed))
print("unchanged        :", len(result.unchanged))
print("errors           :", result.errors)
print("cached after refresh:", engine.execute_sync(query, cache=True).data)

refreshed        : 1
expired_refreshed: 0
unchanged        : 0
errors           : []
cached after refresh: [{'orders.amount_sum': 1350.0}]


## 4. TTL — time-based expiry

A `ttl_seconds` bounds an entry's wall-clock age. It is checked **lazily on read**: once an entry
is older than the TTL, the next `cache=True` read treats it as a miss and re-executes. Here we use
a 1-second TTL and sleep past it.

In [5]:
ttl_engine = SlayerQueryEngine(
    storage=storage,
    cache_config=CacheConfig(ttl_seconds=1.0),
)

print("populate:", ttl_engine.execute_sync(query, cache=True).data)
mutate("INSERT INTO orders VALUES (5, 'pending', 25.0, '2026-02-02')")

print("within TTL (stale):", ttl_engine.execute_sync(query, cache=True).data)
time.sleep(1.1)  # let the entry age past ttl_seconds
print("after TTL  (fresh):", ttl_engine.execute_sync(query, cache=True).data)

populate: [{'orders.amount_sum': 1350.0}]
within TTL (stale): [{'orders.amount_sum': 1350.0}]


after TTL  (fresh): [{'orders.amount_sum': 1375.0}]


## 5. Managing the cache

- `evict(query)` removes one entry (recomputing its key without touching the database).
- `clear_cache()` drops everything.
- `cache_size` reports the live entry count.

In [6]:
print("size before evict:", engine.cache_size)
print("evict returned    :", engine.evict_sync(query))
print("size after evict  :", engine.cache_size)

engine.execute_sync(query, cache=True)
print("size after re-cache:", engine.cache_size)
engine.clear_cache()
print("size after clear   :", engine.cache_size)

size before evict: 1
evict returned    : True
size after evict  : 0
size after re-cache: 1
size after clear   : 0


## Recap

- `execute(query, cache=True)` is opt-in and per-call; `dry_run` / `explain` are never cached.
- The cache is **per engine instance** — two engines (e.g. different tenants / connection settings)
  never share cached rows.
- Staleness has two independent signals: **TTL** (lazy on read) and **refresh keys** (scanned by
  `engine.refresh()`).
- Everything shown here has a `*_sync` wrapper (`execute_sync`, `refresh_sync`, `evict_sync`) and an
  `async` twin.

For the full reference — cache key, table detection, the refresh-key trade-off, and non-goals —
see the [Query Cache concept doc](../../concepts/query-cache.md).